# Sudoku Conceptual Exploration

This notebook demonstrates how **Formal Concept Analysis (FCA)**, **Attribute Exploration**, and **First-Order Rule Exploration** can discover the canonical implications and rules governing Sudoku puzzles.

We investigate two approaches:
1. **Propositional Attribute Exploration (SAT-based)**: Explores value assignments $(r, c, \text{num})$ using **PySAT** as the oracle, accelerated by geometric and algebraic symmetry mappings.
2. **First-Order Relational Rule Exploration (SMT-based)**: Explores multi-sorted first-order relational rules over cell coordinates and number values using the **Z3 SMT solver**.

## 1. Setup & Environment

First, we import core library modules and the Sudoku exploration components.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from conceptual_exploration import AttributeExploration, Implication
from conceptual_exploration.core.theory import ImplicationTheory
from conceptual_exploration.exploration.base import ExplorationBase
from conceptual_exploration.exploration.rule import RuleExploration
from conceptual_exploration.logic.variable import SortedVariable

from explorations.sudoku import (
    SudokuExpert,
    SudokuSort,
    Z3SudokuExpert,
    get_sudoku_attributes,
    get_sudoku_background_implications,
    get_sudoku_predicates,
    get_sudoku_symmetries,
    print_solution,
    solve_sudoku,
    sudoku2sat,
)

print("Sudoku exploration modules loaded successfully.")

## 2. SAT-based Sudoku Reduction & Solving

We reduce Sudoku puzzles into Boolean CNF formulas using **PySAT** and solve them efficiently.

In [ ]:
# Example 4x4 Sudoku (k = 2)
grid_4x4 = [
    [4, 3, 0, 0],
    [2, 0, 0, 3],
    [0, 4, 3, 0],
    [3, 0, 0, 0],
]

solution_4x4 = solve_sudoku(grid_4x4, k=2)
print("Solved 4x4 Sudoku:")
print_solution(solution_4x4)

print("\n" + "="*40 + "\n")

# Example 9x9 Sudoku (k = 3)
grid_9x9 = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]

solution_9x9 = solve_sudoku(grid_9x9, k=3)
print("Solved 9x9 Sudoku:")
print_solution(solution_9x9)

## 3. Propositional Attribute Exploration with Symmetries

In propositional attribute exploration, the attributes are cell entries $(r, c, \text{num})$.

Sudoku exhibits natural symmetry groups:
- **Digit Permutations**: $(n! - 1)$ permutations of numbers
- **Grid Rotations**: 90°, 180°, and 270° clockwise rotations
- **Grid Reflections**: horizontal and vertical reflections

Passing these symmetries to `ExplorationBase` dramatically reduces the number of questions the expert must evaluate.

In [ ]:
k = 2  # 4x4 Sudoku
attributes = get_sudoku_attributes(block_size=k)
symmetries = get_sudoku_symmetries(block_size=k)

print(f"Total attributes (cells x digits): {len(attributes)}")
print(f"Symmetry mappings generated:     {len(symmetries)}")

base = ExplorationBase(
    attributes=attributes,
    mappings=symmetries,
)
expert = SudokuExpert(block_size=k)
exploration = AttributeExploration(base, expert)

# Run exploration
state = exploration.run()

print("\nExploration Summary:")
print(f"  Questions Asked:       {state.questions_asked}")
print(f"  Accepted Base Rules:   {len(base.accepted_implications)}")
print(f"  Total Theory Rules:    {len(base.implications.implications)}")
print(f"  Counterexamples Found: {len(state.counterexamples)}")

## 4. First-Order Relational Rule Exploration with Z3

We now explore relational first-order Sudoku logic over sorted variables:
- Cell variables: $x, y, z \in \text{SudokuSort.CELL}$
- Number variables: $n, m \in \text{SudokuSort.NUMBER}$

The **Z3 SMT solver** serves as the expert oracle evaluating whether implications hold universally across all valid Sudoku grids.

In [ ]:
variables = [
    SortedVariable("x", SudokuSort.CELL),
    SortedVariable("y", SudokuSort.CELL),
    SortedVariable("z", SudokuSort.CELL),
    SortedVariable("n", SudokuSort.NUMBER),
    SortedVariable("m", SudokuSort.NUMBER),
]

k = 2
expert_z3 = Z3SudokuExpert(block_size=k, variables=variables)
predicates = get_sudoku_predicates(block_size=k, expert=expert_z3)

selected_predicates = (
    predicates[0],  # Peers
    predicates[1],  # Apart
    predicates[2],  # Same
    predicates[3],  # Different
    predicates[4],  # Contains
    predicates[5],  # SameNumber
    predicates[6],  # DifferentNumbers
    predicates[8],  # SameRow
    predicates[9],  # SameColumn
)

rule_exploration = RuleExploration(
    selected_predicates,
    variables,
    expert_z3,
    substitutions=True,
    evaluate_all=True,
)

rule_exploration.run()
rule_base = rule_exploration.base

print(f"\nAccepted {len(rule_base.accepted_implications)} relational rules:")
theory = ImplicationTheory(rule_base.implications)
for idx, impl in enumerate(rule_base.accepted_implications[:10], start=1):
    simplified = theory.simplify(impl)
    premise_str = " {" + ", ".join(str(a) for a in simplified.premise) + "}" if simplified.premise else " Ø"
    concl_str = " {" + ", ".join(str(a) for a in simplified.conclusion) + "}"
    print(f"  [{idx}] {premise_str} ==> {concl_str}")